# WGBS Timing Breakdown for `methylseg`

This notebook is a profiling/development example rather than a polished tutorial.
It mirrors the WGBS example workflow, but calls each major step explicitly and records wall-clock timings.

It intentionally uses a couple of semi-internal `MethylSegPathway` methods so we can break apart:

- sample preparation
- `fit_pathway()`
- joint-sample `segment_sample()`
- region creation
- per-chromosome raw BED writing
- cleaned-region generation
- summary-file aggregation


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import os
import shutil
import time

import pandas as pd

from methylseg import HMMType, MethylDataPrep, MethylSegPathway


In [ ]:
ip = get_ipython()
if "__vsc_ipynb_file__" in ip.user_ns:
    NOTEBOOK_PATH = Path(ip.user_ns["__vsc_ipynb_file__"]).resolve()
else:
    NOTEBOOK_PATH = Path("analysis/shared_utils/methylseg/examples/wgbs_timing_breakdown.ipynb").resolve()

EXAMPLES_DIR = NOTEBOOK_PATH.parent
METHYLSEG_ROOT = EXAMPLES_DIR.parent
DATA_DIR = METHYLSEG_ROOT / "data"
REFERENCE_DIR = DATA_DIR / "reference_files"

REFERENCE_FILE = REFERENCE_DIR / "WGBS_colon-primary-tumor_1_wgbs.tsv.gz"
SAMPLE_ID = "WGBS_colon-primary-tumor_1"
MIN_COVERAGE = 10
MIN_PROBES = 5
RESET_OUTPUT_DIR = True
RUN_OPTIONAL_COMPARE = True

OUT_DIR = EXAMPLES_DIR / "out" / "wgbs_timing_breakdown"
COMPARE_OUT_DIR = EXAMPLES_DIR / "out" / "wgbs_timing_breakdown_compare"

REFERENCE_FILE


PosixPath('/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/data/reference_files/WGBS_colon-primary-tumor_1_wgbs.tsv.gz')

In [4]:
timing_rows = []


def run_timed(step_name, func, *args, **kwargs):
    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed_sec = time.perf_counter() - start
    timing_rows.append({"step": step_name, "elapsed_sec": elapsed_sec})
    print(f"{step_name}: {elapsed_sec:,.2f} sec")
    return result


In [5]:
if RESET_OUTPUT_DIR:
    shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_OPTIONAL_COMPARE and RESET_OUTPUT_DIR:
    shutil.rmtree(COMPARE_OUT_DIR, ignore_errors=True)


In [6]:
wgbs_test_sample_info, wgbs_test_sample_info_removed = run_timed(
    "prepare_sample_info",
    lambda: MethylDataPrep(
        meth_file=REFERENCE_FILE,
        sample_id=SAMPLE_ID,
        resolution="wgbs",
        min_coverage=MIN_COVERAGE,
        remove_low_coverage_like_cpgs=True,
    ).prepare(),
)

print(wgbs_test_sample_info)
print(f"Removed CpGs: {len(wgbs_test_sample_info_removed):,}")


prepare_sample_info: 25.92 sec
SampleInfo(sample_id='WGBS_colon-primary-tumor_1', meth_data=        CpG_chrm   CpG_beg   CpG_end      beta
0           chr1     14652     14654  0.416667
1           chr1     14698     14700  0.894737
2           chr1     14709     14711  0.945946
3           chr1     14715     14717  0.828571
4           chr1     14772     14774  0.960000
...          ...       ...       ...       ...
3988097     chrY  56886883  56886885  0.714286
3988098     chrY  56886893  56886895  0.862069
3988099     chrY  56886916  56886918  0.607143
3988100     chrY  56886943  56886945  0.642857
3988101     chrY  56886953  56886955  0.653846

[3988102 rows x 4 columns], resolution='wgbs')
Removed CpGs: 22,795,129


In [7]:
meth_seg_pathway = MethylSegPathway(
    train_sample_info=wgbs_test_sample_info,
    hmm_type=HMMType.STICKY,
    hmm_params={
        "stay_prob": 0.99995,
        "emission_mismatch_prob": 0.45,
        "fit_transitions": False,
    },
    out_dir=OUT_DIR,
)

run_timed("fit_pathway", meth_seg_pathway.fit_pathway)


fit_pathway: 11.86 sec


In [8]:
filtered_sample_info = meth_seg_pathway.subset_sample_info_by_chroms(
    sample_info=wgbs_test_sample_info,
    chroms=None,
)
resolved_chroms = (
    filtered_sample_info.meth_data["CpG_chrm"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

segmented_meth_data, hmm_model = run_timed(
    "segment_sample_joint",
    meth_seg_pathway.segmentor.segment_sample,
    sample_info=filtered_sample_info,
    chrom=None,
    force_resegment=False,
)

print(f"Chromosomes segmented: {len(resolved_chroms)}")
print(f"Segmented CpGs: {len(segmented_meth_data):,}")


segment_sample_joint: 176.66 sec
Chromosomes segmented: 25
Segmented CpGs: 3,988,102


In [9]:
regions_df = run_timed(
    "create_regions",
    meth_seg_pathway.segmentor.create_regions,
    state_col="hmm_state_readable",
    region_min_probes=MIN_PROBES,
)
meth_seg_pathway.segmentor.regions_df = regions_df.copy()

run_timed(
    "write_regions_by_chrom_and_state",
    meth_seg_pathway._write_regions_by_chrom_and_state,
    regions_df=regions_df,
    sample_id=filtered_sample_info.sample_id,
    chroms=resolved_chroms,
)

print(f"Raw regions: {len(regions_df):,}")
regions_df.head()


create_regions: 166.13 sec
write_regions_by_chrom_and_state: 1.14 sec
Raw regions: 134,177


,CpG_chrm,start,end,avg_beta,probe_count,state
0,chr1,14652,19789,0.872591,19,HIGH
1,chr1,56297,63684,0.780385,6,INTERMEDIATE
2,chr1,68238,129333,0.452206,35,PMD
3,chr1,131401,191949,0.812223,131,HIGH
4,chr1,264573,267596,0.375963,6,PMD


In [ ]:
clean_summary_paths, clean_dir = run_timed(
    "get_clean_regions_no_summary",
    meth_seg_pathway.get_clean_regions,
    regions_df=regions_df,
    sample_id=filtered_sample_info.sample_id,
    chrom=None,
    generate_summary_files=False,
)

summary_paths = run_timed(
    "write_summary_files",
    meth_seg_pathway._write_summary_files,
    raw_regions_df=regions_df,
    sample_id=filtered_sample_info.sample_id,
    clean_regions=True,
    chroms=resolved_chroms,
)

print(clean_dir)
summary_paths[:6]


get_clean_regions_no_summary: 2.57 sec
write_summary_files: 1.25 sec
/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown/clean_regions


['/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown/summary_files/segments_raw_LOW.bed',
 '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown/summary_files/segments_raw_PMD.bed',
 '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown/summary_files/segments_raw_INTERMEDIATE.bed',
 '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown/summary_files/segments_raw_HIGH.bed',
 '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown/summary_files/segments_cleaned_LOW.bed',
 '/uufs/chpc.utah.edu/common/home/clement

In [11]:
timing_df = pd.DataFrame(timing_rows)
total_df = pd.DataFrame([
    {"step": "total_tracked_time", "elapsed_sec": timing_df["elapsed_sec"].sum()}
])
timing_summary_df = pd.concat([timing_df, total_df], ignore_index=True)
timing_summary_df


,step,elapsed_sec
0,prepare_sample_info,25.916117
1,fit_pathway,11.855652
2,segment_sample_joint,176.657765
3,create_regions,166.129602
4,write_regions_by_chrom_and_state,1.137630
5,get_clean_regions_no_summary,2.572160
6,write_summary_files,1.251604
7,total_tracked_time,385.520530


## Optional comparison

Set `RUN_OPTIONAL_COMPARE = True` above to run the standard `run_pathway()` workflow into a separate output directory.
This is only for sanity-checking that the manual staged execution matches the real pipeline shape.


In [14]:
if RUN_OPTIONAL_COMPARE:
    compare_pathway = MethylSegPathway(
        train_sample_info=wgbs_test_sample_info,
        hmm_type=HMMType.STICKY,
        hmm_params={
            "stay_prob": 0.99995,
            "emission_mismatch_prob": 0.45,
            "fit_transitions": False,
        },
        out_dir=COMPARE_OUT_DIR,
    )
    compare_region_paths = run_timed(
        "optional_run_pathway_compare",
        compare_pathway.run_pathway,
        sample_info=wgbs_test_sample_info,
        min_probes=MIN_PROBES,
        clean_regions=True,
        verbose=True,
    )
    print(compare_region_paths[:6])
else:
    print("Skipping optional comparison run.")


Fitting pathway...
Generating regions ...
optional_run_pathway_compare: 359.27 sec
['/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown_compare/summary_files/segments_raw_LOW.bed', '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown_compare/summary_files/segments_raw_PMD.bed', '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown_compare/summary_files/segments_raw_INTERMEDIATE.bed', '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/examples/out/wgbs_timing_breakdown_compare/summary_files/segments_raw_HIGH.bed', '/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20230828_tcga_methylation/analysis/shared_utils/methylseg/exampl